# SpendDNA – Complete Minor Project

## 1. Data Parsing and Cleaning

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Data set for DADS June.csv')
df.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [19]:
# Remove duplicates
before=len(df)
df_clean = df.copy()
df_clean=df_clean.drop_duplicates()

# Standardize transaction type
df_clean['Type']=df_clean['Type'].astype(str).str.lower().replace({
    'dr':'debit',
    'cr':'credit'
})

# Clean amount
df_clean['Amount']=(
    df_clean['Amount'].astype(str)
      .str.replace('₹','',regex=False)
      .str.replace('Rs.','',regex=False)
      .str.replace(',','',regex=False)
      .str.strip()
)
df_clean['Amount']=pd.to_numeric(df_clean['Amount'],errors='coerce')

# Parse dates
df_clean['Date']=pd.to_datetime(df_clean['Date'],errors='coerce',dayfirst=True)

# Time features
df_clean['Hour']=df_clean['Time'].astype(str).str[:2].astype(int)
df_clean['Month']=df_clean['Date'].dt.month_name()

print("Duplicates removed:",before-len(df_clean))
print(df_clean.info())

Duplicates removed: 18
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1310 entries, 0 to 1327
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         1310 non-null   datetime64[ns]
 1   Time         1310 non-null   object        
 2   Description  1310 non-null   object        
 3   Type         1310 non-null   object        
 4   Amount       1310 non-null   float64       
 5   Balance      1310 non-null   float64       
 6   Mode         1310 non-null   object        
 7   Ref          1310 non-null   object        
 8   Hour         1310 non-null   int32         
 9   Month        1310 non-null   object        
dtypes: datetime64[ns](1), float64(2), int32(1), object(6)
memory usage: 107.5+ KB
None


## 2. Vendor Extraction

In [20]:
vendor_dict={
'Swiggy Instamart': ['SWIGGY-INSTAMART', 'BUNDL TECH-INSTAMART', 'INSTAMART'],
    'Swiggy':           ['SWIGGY', 'BUNDL'],
    'Zomato Dining':    ['ZOMATO-DINING'],
    'Zomato':           ['ZOMATO'],
    'Zepto':            ['ZEPTO', 'KIRANAKART'],
    'Blinkit':          ['BLINKIT', 'GROFERS'],

    # Groceries
    'BigBasket':        ['BIGBASKET', 'INNOVATIVE RETAIL'],
    'DMart':            ['DMART', 'AVENUE SUPERMARTS'],

    # Ecommerce (Amazon Prime / Nykaa checked before generic Amazon)
    'Amazon Prime':     ['AMAZON-PRIME', 'AMZN PRIME', 'AMAZON PRIME VIDEO'],
    'Nykaa':            ['NYKAA', 'FSN E-COMMERCE'],
    'Amazon':           ['AMAZON', 'AMZN'],
    'Flipkart':         ['FLIPKART', 'FKART'],
    'Myntra':           ['MYNTRA'],

    # Transport
    'Uber':             ['UBER'],
    'Ola':              ['OLA', 'ANI TECHNOLOGIES', 'ROPPEN'],
    'Rapido':           ['RAPIDO'],
    'BMTC':             ['BMTC', 'TUMMOC'],

    # Fuel
    'BPCL':             ['BPCL'],
    'HP Petrol':        ['HP PETROL'],
    'Indian Oil':       ['INDIAN OIL', 'IOC'],

    # Cafe
    'Starbucks':        ['STARBUCKS'],
    'Cafe Coffee Day':  ['COFFEE DAY', 'CCD'],
    'Third Wave Coffee':['THIRDWAVE', 'THIRD WAVE', 'TWC'],

    # Restaurants
    'Truffles':         ['TRUFFLES'],
    'Empire Restaurant':['EMPIRE RESTAURANT'],
    'Meghana Foods':    ['MEGHANA'],
    'Dineout':          ['DINEOUT'],
    'Local Restaurant': ['BANGALORE RESTAURANT', 'UPI-RESTAURANT'],

    # Subscriptions
    'Netflix':          ['NETFLIX'],
    'Spotify':          ['SPOTIFY'],
    'Hotstar':          ['HOTSTAR', 'STAR INDIA'],

    # Utilities
    'Airtel':           ['AIRTEL'],
    'Vodafone Idea':    ['VODAFONE', 'UPI-VI-RECHARGE', 'VI POSTPAID'],
    'Jio':              ['JIO'],
    'BESCOM':           ['BESCOM', 'BANGALORE ELEC SUPPLY'],
    'BWSSB':            ['BWSSB'],

    # Investments
    'Zerodha':          ['ZERODHA'],
    'Groww':            ['GROWW', 'NEXTBILLION'],

    # Entertainment
    'BookMyShow':       ['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE'],

    # Fixed expenses / income
    'Rent':             ['RENT-LANDLORD'],
    'Salary':           ['SALARY'],

    # Special cases (Section 3e of the brief)
    'P2P Transfer':     ['UPI-AMAN', 'UPI-ANKIT', 'UPI-PRIYA', 'UPI-NEHA',
                          'UPI-KARAN', 'UPI-SNEHA', 'UPI-VIKAS'],
    'Cash Withdrawal':  ['ATM-WDL'],
}

def extract_vendor(desc):
    desc=str(desc).upper()
    for v,kws in vendor_dict.items():
        for k in kws:
            if k in desc:
                return v
    return 'Others'

df_clean['vendor_clean']=df_clean['Description'].apply(extract_vendor)
print("Canonical vendors found:", df_clean['vendor_clean'].nunique())
print()
print("Top 10 vendors by transaction count:")
df_clean['vendor_clean'].value_counts().head(10)

Canonical vendors found: 43

Top 10 vendors by transaction count:


Swiggy              176
Ola                 101
Zomato              101
Amazon               76
Zepto                71
Uber                 71
Swiggy Instamart     67
Blinkit              55
Flipkart             47
Starbucks            42
Name: vendor_clean, dtype: int64

## 3. Category Mapping

In [21]:
category_map={
'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Swiggy Instamart': 'Quick Commerce', 'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce',
    'BigBasket': 'Groceries', 'DMart': 'Groceries',
    'Amazon': 'Ecommerce', 'Flipkart': 'Ecommerce', 'Myntra': 'Ecommerce', 'Nykaa': 'Ecommerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'BPCL': 'Fuel', 'HP Petrol': 'Fuel', 'Indian Oil': 'Fuel',
    'Starbucks': 'Cafe', 'Cafe Coffee Day': 'Cafe', 'Third Wave Coffee': 'Cafe',
    'Truffles': 'Restaurants', 'Empire Restaurant': 'Restaurants', 'Meghana Foods': 'Restaurants',
    'Dineout': 'Restaurants', 'Local Restaurant': 'Restaurants', 'Zomato Dining': 'Restaurants',
    'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Hotstar': 'Subscriptions',
    'Amazon Prime': 'Subscriptions',
    'Airtel': 'Utilities', 'Vodafone Idea': 'Utilities', 'Jio': 'Utilities',
    'BESCOM': 'Utilities', 'BWSSB': 'Utilities',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'BookMyShow': 'Entertainment',
    'Rent': 'Rent', 'Salary': 'Income',
    'P2P Transfer': 'Personal Transfer', 'Cash Withdrawal': 'Cash Withdrawal',
}

df_clean['category']=df_clean['vendor_clean'].map(category_map).fillna('Others')
df_clean['category'].value_counts()

Food Delivery        277
Transport            250
Quick Commerce       193
Ecommerce            162
Cafe                  99
Restaurants           93
Utilities             43
Groceries             41
Subscriptions         41
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: category, dtype: int64

## 4. Spending Overview

In [22]:
debit = df_clean[df_clean['Type'] == 'debit']
credit = df_clean[df_clean['Type'] == 'credit']

total_debit = debit['Amount'].sum()
total_credit = credit['Amount'].sum()

net_savings = total_credit - total_debit
savings_rate = (net_savings / total_credit) * 100

top_categories = debit.groupby('category')['Amount'].sum().sort_values(ascending=False).head(5)
top_vendors = debit.groupby('vendor_clean')['Amount'].sum().sort_values(ascending=False).head(5)

print("=" * 70)
print(" " * 20 + "SPENDING OVERVIEW")
print("=" * 70)

print(f"Total Transactions : {len(df)}")
print(f"Total Credits      : ₹{total_credit:,.2f}")
print(f"Total Debits       : ₹{total_debit:,.2f}")
print(f"Net Savings        : ₹{net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")

print("\n" + "-" * 70)
print("Top 5 Spending Categories")
print("-" * 70)

for i, (category, amount) in enumerate(top_categories.items(), start=1):
    print(f"{i}. {category:<25} ₹{amount:>12,.2f}")

print("\n" + "-" * 70)
print("Top 5 Vendors")
print("-" * 70)

for i, (vendor, amount) in enumerate(top_vendors.items(), start=1):
    print(f"{i}. {vendor:<25} ₹{amount:>12,.2f}")

print("=" * 70)

                    SPENDING OVERVIEW
Total Transactions : 1328
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%

----------------------------------------------------------------------
Top 5 Spending Categories
----------------------------------------------------------------------
1. Ecommerce                 ₹  593,769.00
2. Investments               ₹  248,160.00
3. Restaurants               ₹  127,290.00
4. Food Delivery             ₹  119,501.00
5. Rent                      ₹  108,000.00

----------------------------------------------------------------------
Top 5 Vendors
----------------------------------------------------------------------
1. Amazon                    ₹  318,422.00
2. Zerodha                   ₹  210,000.00
3. Flipkart                  ₹  177,510.00
4. Rent                      ₹  108,000.00
5. Swiggy                    ₹   73,738.00


## 5. Monthly Trend

In [23]:
pivot=debit.pivot_table(values='Amount',index='category',columns='Month',aggfunc='sum',fill_value=0)
pivot

Month,April,February,January,June,March,May
category,,,,,,
Cafe,6564,4273,3690,5802,5448,5668
Cash Withdrawal,5500,5000,2000,17000,8000,8000
Ecommerce,68098,92773,97134,136991,103772,95001
Entertainment,2224,474,1263,1914,2418,0
Food Delivery,20726,18714,20206,19496,19166,21193
Fuel,18718,2079,30322,2882,26164,9138
Groceries,11748,8571,17649,5432,6289,9718
Investments,54126,15000,38432,23330,68644,48628
Personal Transfer,663,4285,7852,4112,4275,3412


## 6. Time of Day Analysis

In [24]:
print("\nTIME-OF-DAY PATTERNS")

# ---------- Food Delivery ----------
food = debit[debit['category'] == 'Food Delivery']

late_food = food[
    (food['Hour'] >= 21) | (food['Hour'] <= 1)
]

food_pct = len(late_food) / len(food) * 100 if len(food) else 0

print(f" Food Delivery peaks : 21:00 - 01:00 ({food_pct:.1f}% of orders)")


# ---------- Cafe ----------
cafe = debit[debit['category'] == 'Cafe']

if len(cafe):
    morning = cafe[(cafe['Hour'] >= 9) & (cafe['Hour'] <= 11)]
    morning_pct = len(morning) / len(cafe) * 100

    print(f" Cafe peaks          : 09:00 - 11:00 ({morning_pct:.1f}% of cafe visits)")
else:
    print(" Cafe peaks          : No cafe transactions")


# ---------- Quick Commerce ----------
quick = debit[debit['category'] == 'Quick Commerce']

if len(quick):

    hourly = quick.groupby('Hour').size()
    cv = hourly.std() / hourly.mean()

    if cv < 0.5:
        print(" Quick Commerce      : Evenly distributed")
    else:
        peak_hour = hourly.idxmax()
        print(f" Quick Commerce      : Peaks around {peak_hour:02d}:00")

else:
    print(" Quick Commerce      : No transactions")
hour_matrix=debit.pivot_table(values='Amount',index='category',columns='Hour',aggfunc='count',fill_value=0)
hour_matrix




TIME-OF-DAY PATTERNS
 Food Delivery peaks : 21:00 - 01:00 (19.9% of orders)
 Cafe peaks          : 09:00 - 11:00 (26.3% of cafe visits)
 Quick Commerce      : Peaks around 20:00


Hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
category,,,,,,,,,,,,,,,,,,,,,
Cafe,3,3,0,1,0,0,1,2,9,7,...,5,7,8,10,7,2,5,0,0,2
Cash Withdrawal,0,0,0,0,0,0,0,0,2,3,...,2,0,1,0,1,0,1,1,2,0
Ecommerce,9,5,4,7,7,7,6,4,4,9,...,9,5,8,7,9,4,7,7,6,9
Entertainment,1,1,0,0,0,0,0,1,2,0,...,0,0,2,0,2,1,2,0,1,0
Food Delivery,1,7,2,4,6,7,1,1,7,10,...,9,13,9,10,27,29,34,19,19,9
Fuel,2,1,0,1,1,1,0,0,1,2,...,1,2,3,0,1,2,1,2,0,1
Groceries,2,1,2,6,1,0,4,0,1,4,...,2,0,0,0,1,1,2,3,0,1
Investments,1,1,0,0,2,1,4,0,0,1,...,0,0,0,0,1,1,1,0,1,2
Personal Transfer,0,0,0,0,0,0,0,0,1,1,...,1,0,2,3,1,1,1,3,2,0


## 7. Anomaly Detection

In [25]:
mean=df_clean.groupby('category')['Amount'].transform('mean')
std=df_clean.groupby('category')['Amount'].transform('std')

df_clean['z_score']=(df_clean['Amount']-mean)/std

anomalies=df_clean[df_clean['z_score']>2].sort_values('z_score',ascending=False)
print("Top 10 anomalies:")
anomalies[['Date','vendor_clean','category','Amount','z_score']].head(10)

Top 10 anomalies:


,Date,vendor_clean,category,Amount,z_score
414,2024-02-26,Local Restaurant,Restaurants,8383.0,4.348108
1271,2024-06-22,Dineout,Restaurants,7935.0,4.070396
653,2024-03-31,Meghana Foods,Restaurants,7931.0,4.067916
1298,2024-06-26,Amazon,Ecommerce,22008.0,3.974482
269,2024-02-07,Amazon,Ecommerce,21986.0,3.969715
464,2024-03-04,Truffles,Restaurants,7441.0,3.764169
49,2024-01-07,Local Restaurant,Restaurants,7314.0,3.685442
475,2024-03-05,Amazon,Ecommerce,19917.0,3.521407
450,2024-03-02,Amazon,Ecommerce,18273.0,3.165188
1079,2024-05-27,Flipkart,Ecommerce,17831.0,3.069416


## 8. Spending Archetype Detection

In [26]:


debit_total = debit['Amount'].sum()

# Percentages
food_pct = (
    debit[debit['category'].isin(['Food Delivery', 'Restaurants', 'Cafe'])]['Amount'].sum()
    / debit_total * 100
)

quick_pct = (
    debit[debit['category'] == 'Quick Commerce']['Amount'].sum()
    / debit_total * 100
)

shop_pct = (
    debit[debit['category'] == 'Ecommerce']['Amount'].sum()
    / debit_total * 100
)

invest_pct = (
    debit[debit['category'] == 'Investments']['Amount'].sum()
    / debit_total * 100
)

transport_pct = (
    debit[debit['category'] == 'Transport']['Amount'].sum()
    / debit_total * 100
)

# Late-night food delivery
food_delivery = debit[debit['category'] == 'Food Delivery']

late_night_pct = (
    food_delivery[
        (food_delivery['Hour'] >= 21) |
        (food_delivery['Hour'] <= 2)
    ].shape[0]
    / len(food_delivery)
    * 100
)

# Active subscription vendors
subscription_count = (
    debit[debit['category'] == 'Subscriptions']['vendor_clean']
    .nunique()
)

# ----------------------------
# Print Results
# ----------------------------

print("=" * 65)
print(" " * 20 + "SPENDING ARCHETYPES")
print("=" * 65)

print(f"[{'YES' if food_pct > 25 else 'NO '}] THE FOODIE                    ({food_pct:.1f}% on food)")
print(f"[{'YES' if quick_pct > 15 else 'NO '}] THE QUICK COMMERCE JUNKIE     ({quick_pct:.1f}% on Q-commerce)")
print(f"[{'YES' if shop_pct > 15 else 'NO '}] THE SHOPAHOLIC                ({shop_pct:.1f}% on e-commerce)")
print(f"[{'YES' if invest_pct > 15 else 'NO '}] THE INVESTOR                  ({invest_pct:.1f}% on SIPs)")
print(f"[{'YES' if late_night_pct > 50 else 'NO '}] THE LATE-NIGHT SNACKER       ({late_night_pct:.1f}% food after 9 PM)")
print(f"[{'YES' if transport_pct > 10 else 'NO '}] THE CAB COMMUTER             ({transport_pct:.1f}% on transport)")
print(f"[{'YES' if subscription_count >= 5 else 'NO '}] THE SUBSCRIPTION LOVER      ({subscription_count} active subscriptions)")
print(f"[{'YES' if savings_rate < 10 else 'NO '}] THE YOLO SPENDER             (Savings Rate {savings_rate:.1f}%)")
print(f"[{'YES' if savings_rate > 40 else 'NO '}] THE DISCIPLINED SAVER        (Savings Rate {savings_rate:.1f}%)")

print("=" * 65)

                    SPENDING ARCHETYPES
[NO ] THE FOODIE                    (16.6% on food)
[NO ] THE QUICK COMMERCE JUNKIE     (5.7% on Q-commerce)
[YES] THE SHOPAHOLIC                (35.4% on e-commerce)
[NO ] THE INVESTOR                  (14.8% on SIPs)
[NO ] THE LATE-NIGHT SNACKER       (20.6% food after 9 PM)
[NO ] THE CAB COMMUTER             (3.4% on transport)
[NO ] THE SUBSCRIPTION LOVER      (4 active subscriptions)
[YES] THE YOLO SPENDER             (Savings Rate -229.3%)
[NO ] THE DISCIPLINED SAVER        (Savings Rate -229.3%)


## 9. Final Report

In [27]:
print("=" * 64)
print(" SpendDNA REPORT - RAHUL SHARMA")
print(f" 6 months - {len(df_clean):,} transactions - Jan to Jun 2024")
print("=" * 64)

# Executive Summary

print(" EXECUTIVE SUMMARY")

net_change = total_credit - total_debit
status = "SURPLUS" if net_change >= 0 else "OVERSPENDING"

print(f" Total Credits : Rs. {total_credit:,.0f}")
print(f" Total Debits  : Rs. {total_debit:,.0f}")
print(f" Net Change    : {'-' if net_change < 0 else ''}Rs. {abs(net_change):,.0f} ({status})")
print(f" Savings Rate  : {savings_rate:.1f}%")

print(f" Transactions  : {len(df_clean):,}")
print(f" Unique Vendors: {df_clean['vendor_clean'].nunique()}")

# Top Categories

print("\n TOP CATEGORIES (% of debit total)")

category_totals = debit.groupby('category')['Amount'].sum().sort_values(ascending=False)
category_pct = category_totals / category_totals.sum() * 100

for category in category_totals.head(5).index:

    amount = category_totals[category]
    pct = category_pct[category]

    bar = "#" * int(pct)

    print(f" {category:<18} {bar:<22} {pct:5.1f}%  Rs. {amount:,.0f}")

# Top Vendors

print("\n TOP VENDORS")

vendor_amount = debit.groupby('vendor_clean')['Amount'].sum().sort_values(ascending=False)
vendor_count = debit['vendor_clean'].value_counts()

for vendor in vendor_amount.head(5).index:

    print(f" {vendor:<20} Rs. {vendor_amount[vendor]:>9,.0f} ({vendor_count[vendor]:>3} transactions)")

# Time of Day

print("\n TIME-OF-DAY PATTERNS")
print(f" Food Delivery peaks : 21:00 - 01:00 ({food_pct:.1f}% of orders)")
print(f" Cafe peaks          : 09:00 - 11:00 ({morning_pct:.1f}% of cafe visits)")
print(f" Quick Commerce      : Peaks around {peak_hour:02d}:00")

# Monthly Trend

print("\n MONTHLY FOOD DELIVERY TREND")

food_month = (
    debit[debit['category']=="Food Delivery"]
    .groupby(debit['Date'].dt.strftime('%b'))['Amount']
    .sum()
)

for month, amount in food_month.items():

    bar = "#" * int(amount / food_month.max() * 15)

    print(f" {month:<3} Rs. {amount:>8,.0f} {bar}")

# Anomalies

print("\n TOP ANOMALIES")

top_anomalies = anomalies.sort_values("z_score",ascending=False).head(5)

for _, row in top_anomalies.iterrows():

    print(
        f" {row['Date'].strftime('%d %b')} - "
        f"{row['vendor_clean']:<15} "
        f"Rs. {row['Amount']:>8,.0f} "
        f"(z={row['z_score']:.1f})"
    )

# Archetypes

print("\n RAHUL'S SPENDING ARCHETYPES")

print(f"[{'YES' if food_pct > 25 else 'NO '}] THE FOODIE                    ({food_pct:.1f}% on food)")
print(f"[{'YES' if quick_pct > 15 else 'NO '}] THE QUICK COMMERCE JUNKIE     ({quick_pct:.1f}% on Q-commerce)")
print(f"[{'YES' if shop_pct > 15 else 'NO '}] THE SHOPAHOLIC                ({shop_pct:.1f}% on e-commerce)")
print(f"[{'YES' if invest_pct > 15 else 'NO '}] THE INVESTOR                  ({invest_pct:.1f}% on SIPs)")
print(f"[{'YES' if late_night_pct > 50 else 'NO '}] THE LATE-NIGHT SNACKER       ({late_night_pct:.1f}% food after 9 PM)")
print(f"[{'YES' if transport_pct > 10 else 'NO '}] THE CAB COMMUTER             ({transport_pct:.1f}% on transport)")
print(f"[{'YES' if subscription_count >= 5 else 'NO '}] THE SUBSCRIPTION LOVER      ({subscription_count} active subscriptions)")
print(f"[{'YES' if savings_rate < 10 else 'NO '}] THE YOLO SPENDER             (Savings Rate {savings_rate:.1f}%)")
print(f"[{'YES' if savings_rate > 40 else 'NO '}] THE DISCIPLINED SAVER        (Savings Rate {savings_rate:.1f}%)")

# Insights

print("\n" + "=" * 64)
print(" KEY INSIGHTS")

monthly_overspend = abs(net_change) / 6

print(f" 1. Rahul is overspending by about Rs. {monthly_overspend:,.0f} per month.")

top_category = category_totals.index[0]

print(f" 2. '{top_category}' is Rahul's largest spending category "
      f"({category_pct.iloc[0]:.1f}% of all debit spending).")

print(f" 3. Rahul transacted with {df_clean['vendor_clean'].nunique()} unique vendors "
      f"across {df_clean['category'].nunique()} categories.")

print("=" * 64)

 SpendDNA REPORT - RAHUL SHARMA
 6 months - 1,310 transactions - Jan to Jun 2024
 EXECUTIVE SUMMARY
 Total Credits : Rs. 509,774
 Total Debits  : Rs. 1,678,901
 Net Change    : -Rs. 1,169,127 (OVERSPENDING)
 Savings Rate  : -229.3%
 Transactions  : 1,310
 Unique Vendors: 43

 TOP CATEGORIES (% of debit total)
 Ecommerce          ###################################  35.4%  Rs. 593,769
 Investments        ##############          14.8%  Rs. 248,160
 Restaurants        #######                  7.6%  Rs. 127,290
 Food Delivery      #######                  7.1%  Rs. 119,501
 Rent               ######                   6.4%  Rs. 108,000

 TOP VENDORS
 Amazon               Rs.   318,422 ( 76 transactions)
 Zerodha              Rs.   210,000 ( 14 transactions)
 Flipkart             Rs.   177,510 ( 47 transactions)
 Rent                 Rs.   108,000 (  6 transactions)
 Swiggy               Rs.    73,738 (176 transactions)

 TIME-OF-DAY PATTERNS
 Food Delivery peaks : 21:00 - 01:00 (16.6% of or